# GraphEm Rapids quick start

This notebook is a small, CPU-safe package tutorial. The separate [`fast-geometric-repro`](https://github.com/sashakolpakov/fast-geometric-repro) repository owns the pinned H100 capacity, ANN, stress, and influence-reproduction workflow.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

working_directory = Path.cwd()
source_root = (
    working_directory.parent
    if not (working_directory / 'graphem_rapids').is_dir()
    and (working_directory.parent / 'graphem_rapids').is_dir()
    else working_directory
)
if (source_root / 'graphem_rapids').is_dir():
    sys.path.insert(0, str(source_root))
if importlib.util.find_spec('graphem_rapids') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'git+https://github.com/sashakolpakov/graphem-rapids.git@main',
    ])

import numpy as np
import graphem_rapids as gr

print(gr.__version__)
print(gr.get_backend_info())

## Embed a small graph

The explicit CPU backend keeps this notebook runnable without CUDA. Production code can use `backend='auto'`; 2D is the geometric crossing-force mode.

In [ ]:
adjacency = gr.generate_er(n=80, p=0.06, seed=7)
embedder = gr.create_graphem(
    adjacency=adjacency,
    n_components=2,
    backend='pytorch',
    device='cpu',
    sample_size=min(64, adjacency.nnz // 2),
    batch_size=64,
    verbose=False,
)
embedder.run_layout(num_iterations=3)
positions = embedder.get_positions()
assert positions.shape == (80, 2)
assert np.isfinite(positions).all()
positions[:5]

## Select active seeds and evaluate repeated cascades

Selection does not run hidden layout iterations. The two methods below are evaluated in identical counter-based live-edge worlds.

In [ ]:
graph_seeds = gr.graphem_seed_selection(embedder, k=5)
degree_discount_seeds = gr.degree_discount_seed_selection(
    adjacency, k=5, p=0.05
)
graph_spread = gr.estimate_independent_cascade(
    adjacency, graph_seeds, p=0.05, n_simulations=32,
    random_seed=11, backend='cpu',
)
degree_discount_spread = gr.estimate_independent_cascade(
    adjacency, degree_discount_seeds, p=0.05, n_simulations=32,
    random_seed=11, backend='cpu',
)
{
    'graphem': {'seeds': graph_seeds, 'mean_spread': graph_spread.mean},
    'degree_discount': {
        'seeds': degree_discount_seeds,
        'mean_spread': degree_discount_spread.mean,
    },
}